# README
- Author: RUEI-HSIANG, CHANG (B12303030)，Codex協助
- Created At: 2026-10-19
- Last Modified At: 2026-10-19
- Purpose: 分離13位高雄候選人的2022官方公報欄位，使用已視讀的固定座標；回查31個公民團體人物頁，排除同名不同縣市、占位資料及不完整履歷，補來源與社群入口。
- Input: 固定hash之pre_bulletin_round5快照五表、保存的整頁公報、歷史待核對名錄與PrideWatch來源manifest／計畫。逐檔路徑與hash見PINS。
- Output: council_bulletin_cells_round5、pridewatch_profile_review_round5、candidate_background_evidence_round5與更新的5份current資料。
- Verification: 固定原頁hash、表格座標、姓名／生日／性別視讀核對值、原資料庫相符但可能同源；只將兩個有另行履歷／當屆情境支持者接回歷史支持欄，其他候選值分欄保存。未知不填零；學校與照片不作性別推測。
- Known limits: 公報為歷史自填資料，不是戶籍或學位認證；公民團體頁為候選人申報／歷史紀錄，含過期與占位內容。社群為來源連結，目標帳號內容尚未全面核對，貼文與追蹤數仍依指示暫緩。


版本註記：第五輪輸出已封存至 `data/processed/snapshots/pre_career_round6_20261019/`。本文件驗證封存版本；最新current由第六輪產生。

In [1]:
from pathlib import Path
import pandas as pd,json,re,hashlib,datetime,unicodedata
from bs4 import BeautifulSoup
from urllib.parse import urlparse,urlunparse,parse_qs,urlencode
import pypdfium2 as pdfium
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def read(p):return pd.read_csv(ROOT/p,dtype=str).fillna('')
def norm(s):return re.sub(r'\s+','',unicodedata.normalize('NFKC',str(s))).replace('台','臺').replace('温','溫')
PINS={'data/processed/snapshots/pre_bulletin_round5_20261019/candidate_master_current.csv': 'df9c43800b4e67b5153ead90ad52128b922d3773c4285e3a3c079f059dbed746', 'data/processed/snapshots/pre_bulletin_round5_20261019/candidate_fields_current.csv': '3ac72c7a54ec9e6aaa0ef8c3a0501d415fccefd2537e51d778540434b84513dc', 'data/processed/snapshots/pre_bulletin_round5_20261019/council_background_20261019.csv': '46dbe63f4d57ef57efba0497d4d3e906c59397b738a6f4b40691af43a8243256', 'data/processed/snapshots/pre_bulletin_round5_20261019/candidate_social_account_evidence_current.csv': '9178c49adc786db50c9865c2d98ec11747cab0a8cf692dd53c14c3033630649a', 'data/processed/snapshots/pre_bulletin_round5_20261019/meta_candidate_account_worklist_current.csv': '665fd1c45bbb06a312aaa28d9868ba20d3a7361c77fb411801c37158684a23ec', 'data/processed/candidate_background_source_pages_20261019.csv': 'a1e32f594487be152fbc2251749693cb8dabb7468c7f8ec4176978661fe1c7da', 'data/processed/council_history_pending_identity_20261019.csv': 'e88d1bca609d0a837dfc597e95d10d85a1dec12a5feaf8d542c6fc25388c143f', 'data/raw/pridewatch_directory_20261019/manifest.json': 'd04a77c63b9b4359771ee198df6c753f7a1939d740bf549fe7bcd41e443a4f0a', 'data/raw/pridewatch_directory_20261019/retrieval_plan.json': 'efb14b163f90b7557f5520367c869d20599a5c1dc460228dd6e9bc901887397f'}
for p,h in PINS.items():assert sha(ROOT/p)==h,p
B='data/processed/snapshots/pre_bulletin_round5_20261019/'
master=read(B+'candidate_master_current.csv');fields=read(B+'candidate_fields_current.csv');council=read(B+'council_background_20261019.csv');accounts=read(B+'candidate_social_account_evidence_current.csv');oldwork=read(B+'meta_candidate_account_worklist_current.csv')
old_f=fields.copy();old_c=council.copy();old_m=master.copy();old_a=accounts.copy()
pages=read('data/processed/candidate_background_source_pages_20261019.csv');history=read('data/processed/council_history_pending_identity_20261019.csv')
manifest=json.loads((ROOT/'data/raw/pridewatch_directory_20261019/manifest.json').read_text());plan=json.loads((ROOT/'data/raw/pridewatch_directory_20261019/retrieval_plan.json').read_text())['profiles'];byurl={r['url']:r for r in manifest}
for r in manifest:
 if r['path']:assert sha(ROOT/r['path'])==r['sha256']
CELL_REVIEW=[{'candidate_key': '2026|直轄市議員|高雄市第1選舉區|林慧欣', 'candidate_name': '林慧欣', 'row_top': 1020.196, 'row_bottom': 1360.418, 'x_shift': 0, 'expected_birth': '74年6月11日', 'expected_sex': '女'}, {'candidate_key': '2026|直轄市議員|高雄市第2選舉區|李柏融', 'candidate_name': '李柏融', 'row_top': 1378.436, 'row_bottom': 1724.666, 'x_shift': 0, 'expected_birth': '38年11月2日', 'expected_sex': '男'}, {'candidate_key': '2026|直轄市議員|高雄市第2選舉區|鄭東元', 'candidate_name': '鄭東元', 'row_top': 1032.202, 'row_bottom': 1378.436, 'x_shift': 0, 'expected_birth': '48年4月20日', 'expected_sex': '男'}, {'candidate_key': '2026|直轄市議員|高雄市第4選舉區|柳淑芳', 'candidate_name': '柳淑芳', 'row_top': 1378.436, 'row_bottom': 1724.666, 'x_shift': 0, 'expected_birth': '49年8月30日', 'expected_sex': '女'}, {'candidate_key': '2026|直轄市議員|高雄市第5選舉區|陳若翠', 'candidate_name': '陳若翠', 'row_top': 685.972, 'row_bottom': 1032.202, 'x_shift': 0, 'expected_birth': '58年4月29日', 'expected_sex': '女'}, {'candidate_key': '2026|直轄市議員|高雄市第5選舉區|陳惠君', 'candidate_name': '陳惠君', 'row_top': 1032.202, 'row_bottom': 1378.436, 'x_shift': 0, 'expected_birth': '57年9月4日', 'expected_sex': '女'}, {'candidate_key': '2026|直轄市議員|高雄市第7選舉區|童燕珍', 'candidate_name': '童燕珍', 'row_top': 1724.666, 'row_bottom': 2074.46, 'x_shift': -0.75, 'expected_birth': '42年11月2日', 'expected_sex': '女'}, {'candidate_key': '2026|直轄市議員|高雄市第7選舉區|林于凱', 'candidate_name': '林于凱', 'row_top': 1724.666, 'row_bottom': 2074.907, 'x_shift': -0.75, 'expected_birth': '70年4月12日', 'expected_sex': '男'}, {'candidate_key': '2026|直轄市議員|高雄市第9選舉區|蘇致榮', 'candidate_name': '蘇致榮', 'row_top': 339.743, 'row_bottom': 685.972, 'x_shift': 0, 'expected_birth': '66年10月29日', 'expected_sex': '男'}, {'candidate_key': '2026|直轄市議員|高雄市第9選舉區|徐尚賢', 'candidate_name': '徐尚賢', 'row_top': 1378.436, 'row_bottom': 1724.666, 'x_shift': 0, 'expected_birth': '79年9月30日', 'expected_sex': '男'}, {'candidate_key': '2026|直轄市議員|高雄市第10選舉區|黃敬雅', 'candidate_name': '黃敬雅', 'row_top': 1378.436, 'row_bottom': 1724.666, 'x_shift': 0, 'expected_birth': '77年10月26日', 'expected_sex': '女'}, {'candidate_key': '2026|直轄市議員|高雄市第10選舉區|湯茹婷', 'candidate_name': '湯茹婷', 'row_top': 1032.202, 'row_bottom': 1378.436, 'x_shift': 0, 'expected_birth': '82年11月19日', 'expected_sex': '女'}, {'candidate_key': '2026|直轄市議員|高雄市第11選舉區|洪村銘', 'candidate_name': '洪村銘', 'row_top': 1032.202, 'row_bottom': 1378.436, 'x_shift': 0, 'expected_birth': '67年11月22日', 'expected_sex': '男'}]
# These precise boxes were located from the official ruled table and visually read before acceptance.
# Font outlines are not used as table edges. Education/experience end before the policy column.
cells=[];evidence=[]
def ev(key,name,field,value,url,file,filehash,locator,year,status):
 evidence.append(dict(candidate_key=key,candidate_name=name,field=field,value_raw=value,source_url=url,source_file=file,source_sha256=filehash,source_locator=locator,source_year=year,identity_status=status,checked_date='2026-10-19'))
for q in CELL_REVIEW:
 d=pages[pages.candidate_key.eq(q['candidate_key'])];assert len(d)==1;r=d.iloc[0].to_dict();assert sha(ROOT/r['source_file'])==r['source_sha256']
 pdf=pdfium.PdfDocument(ROOT/r['source_file']);page=pdf[int(r['page'])-1];tp=page.get_textpage();height=page.get_height()
 cols={'name_cell':(301,362),'birth_roc_raw':(362,402.5),'sex_raw':(402.5,442.8),'education_raw':(523.5,677.5),'experience_raw':(678,858.7)}
 row={k:tp.get_text_bounded(left=x0+q['x_shift']+1,bottom=height-q['row_bottom']+1,right=x1+q['x_shift']-1,top=height-q['row_top']-1).replace('\r','').replace('\n','\\n') for k,(x0,x1) in cols.items()}
 # Source text contains real newline characters; preserve them as literal \n in one physical CSV record.
 row={k:v.replace('\r','').replace('\n','\\n') for k,v in row.items()}
 compact=lambda x:norm(x.replace('\\n','').replace('\n','').replace('\r',''))
 assert compact(row['name_cell'])==q['candidate_name'],row
 assert compact(row['birth_roc_raw'])==q['expected_birth'] and row['sex_raw'].strip()==q['expected_sex'],row
 match=re.fullmatch(r'(\d+)年(\d+)月(\d+)日',compact(row['birth_roc_raw']));assert match
 y,month,day=map(int,match.groups());dob=datetime.date(y+1911,month,day).isoformat()
 row.update({k:r[k] for k in ['candidate_key','candidate_name','county_city','source_url','source_file','source_sha256','page']});row.update(q);row.update(birth_date_declared=dob,birth_year_declared=str(y+1911),source_year='2022',identity_status='2022_bulletin_candidate_cells_read_2026_identity_link_pending')
 h=history[history.candidate_key.eq(r['candidate_key'])];assert len(h)==1;h=h.iloc[0]
 assert h.birth_date_history_candidate==dob and h.history_sex==q['expected_sex']
 row['history_same_origin_note']='與2022官方資料庫相符，但可能同源，不當作第二份獨立身分證據。';row['same_district_number']=h.same_district_number
 for field,value in [('birth_date',dob),('sex',q['expected_sex']),('education',row['education_raw']),('experience',row['experience_raw'])]:ev(r['candidate_key'],r['candidate_name'],field,value,r['source_url'],r['source_file'],r['source_sha256'],f"PDF page {r['page']}; reviewed row top={q['row_top']}, bottom={q['row_bottom']}; fixed {field} column",'2022',row['identity_status'])
 cells.append(row);tp.close();page.close();pdf.close()
cells=pd.DataFrame(cells);assert len(cells)==13 and cells.candidate_key.is_unique
# Parse only explicit profile-specific fields; footer, questionnaires and other candidates are excluded.
profiles=[];social_candidates=[]
legacy_accept={'林柏勛','陳鶴文','陳珍裕','林志豪','黃朝淵','黃敬雅'}
for name,url in plan.items():
 r=byurl[url];assert r['status']=='200';s=BeautifulSoup((ROOT/r['path']).read_bytes().decode('utf8'),'html.parser')
 def f(cls):
  n=s.select_one('.'+cls+' .field-content');return n.get_text(' | ',strip=True).replace('\r','').replace('\n','\\n') if n else ''
 role=f('views-field-field-election-position-2026');area=f('views-field-field-area-2026');ed=f('views-field-field-education');ex=f('views-field-field-qualifications');hist=f('views-field-field-election-history')
 persons=master[master.candidate_name.eq(name)];assert len(persons)==1;p=persons.iloc[0].to_dict()
 status='historical_profile_current_county_context_unconfirmed';accepted=False
 if norm(name) not in norm(f('views-field-field-name')):status='page_without_matching_candidate_specific_name_field'
 elif '啦啦啦' in ed+ex:status='excluded_placeholder_content'
 elif name=='陳冠宇':
  assert '新竹縣' in ex+hist and p['county_city']=='高雄市';status='excluded_same_name_different_county_without_identity_link'
 elif not ed and not ex:status='no_background_fields_present'
 elif role and '2026' in role:
  county=norm(p['county_city']);a=norm(area);district=re.search(r'第(\d+)選舉區',p['electoral_district']);assert district
  assert a.startswith(county+district.group(1)+'-') and '議員' in role,(name,a,p)
  status='2026_profile_name_county_district_match_not_degree_certification';accepted=True
 elif name in legacy_accept:
  assert norm(p['county_city']) in norm(ed+ex+hist) or name in ['林柏勛','黃敬雅']
  status='historical_profile_county_or_distinctive_biography_match_current_fields_pending';accepted=True
 row=dict(candidate_key=p['candidate_key'],candidate_name=name,county_city=p['county_city'],profile_url=url,role_2026=role,area_2026=area,education_raw=ed,experience_raw=ex,election_history_raw=hist,source_year='2026' if role else '2022_or_earlier_not_currently_reconfirmed',identity_status=status,accepted_for_source_inventory=str(accepted),source_file=r['path'],source_sha256=r['sha256'])
 profiles.append(row)
 if accepted:
  for field,value in [('education',ed),('experience',ex)]:
   if value:ev(p['candidate_key'],name,field,value,url,r['path'],r['sha256'],'.views-field-field-'+('education' if field=='education' else 'qualifications')+' .field-content',row['source_year'],status)
  for a in s.select('ul.candidate-social-links a[href]'):social_candidates.append(dict(candidate_key=p['candidate_key'],candidate_name=name,county_city=p['county_city'],office_type=p['office_type'],url=a['href'],source_url=url,source_file=r['path'],source_year=row['source_year'],source_identity_status=status))
profiles=pd.DataFrame(profiles);assert len(profiles)==31 and profiles.candidate_key.is_unique
# Current-world corroboration of Huang's distinctive career; archive the media page itself.
media=byurl['https://finance.ftvnews.com.tw/news/detail/2026305W0223'];text=BeautifulSoup((ROOT/media['path']).read_bytes().decode('utf8'),'html.parser').get_text(' ',strip=True)
assert '黃敬雅' in text and '2026' in text and '復興航空' in text and '前鎮' in text
# Update the inventory while retaining the source tier. The 11 other birth dates are candidate values, not supported identities.
def update(key,field,value,url,status,note):
 mask=fields.candidate_key.eq(key)&fields.field.eq(field);assert mask.sum()==1
 fields.loc[mask,['existing_value','evidence','status','note']]=[value,url,status,note]
 master.loc[master.candidate_key.eq(key),field+'_status']=status
new_profiles=[]
for r in profiles[profiles.accepted_for_source_inventory.eq('True')].to_dict('records'):
 mask=council.candidate_key.eq(r['candidate_key']);assert mask.sum()==1
 p=council[mask].iloc[0]
 if p.moi_profile_url=='' and p.campaign_profile_url=='' and p.additional_profile_url=='':
  new_profiles.append(r['candidate_key']);council.loc[mask,['additional_profile_url','additional_profile_status']]=[r['profile_url'],r['identity_status']]
  update(r['candidate_key'],'profile',r['profile_url'],r['profile_url'],'已保存人物背景原文；年代與身分狀態分列',r['identity_status'])
 if p.education_raw=='' and r['education_raw']:
  council.loc[mask,['education_raw','background_source_url']]=[r['education_raw'],r['profile_url']]
  update(r['candidate_key'],'education',r['education_raw'],r['profile_url'],'已取得公民團體人物頁申報原文；學位未認證',r['source_year']+'；在學／就讀中不當作畢業。')
 if p.experience_raw=='' and r['experience_raw']:council.loc[mask,'experience_raw']=r['experience_raw']
newcols=['birth_date_bulletin_candidate','birth_year_bulletin_candidate','sex_bulletin_candidate','education_bulletin_raw','experience_bulletin_raw','bulletin_source_url','bulletin_source_file','bulletin_source_page','bulletin_identity_status']
for col in newcols:
 assert col not in council;council[col]=''
for r in cells.to_dict('records'):
 mask=council.candidate_key.eq(r['candidate_key']);council.loc[mask,newcols]=[r['birth_date_declared'],r['birth_year_declared'],r['sex_raw'].strip(),r['education_raw'],r['experience_raw'],r['source_url'],r['source_file'],r['page'],r['identity_status']]
 if r['candidate_name'] not in ['林于凱','黃敬雅']:
  for field in ['birth_date','sex','education']:
   maskf=fields.candidate_key.eq(r['candidate_key'])&fields.field.eq(field)
   if fields.loc[maskf,'existing_value'].eq('').all():
    fields.loc[maskf,['status','evidence','note']]=['已分離2022公報欄位；2026身分連結待核對',r['source_url'],'候選值另存議員背景表bulletin欄；未填成已核實值，與歷史資料庫可能同源。']
    master.loc[master.candidate_key.eq(r['candidate_key']),field+'_status']='已分離2022公報欄位；2026身分連結待核對'
# Two explicit identity linkages: Lin's 2026 profile matches two degrees and prior office;
# Huang's school sequence and airline career match, and 2026 media connects that career to the current election.
for name in ['林于凱','黃敬雅']:
 r=cells[cells.candidate_name.eq(name)].iloc[0].to_dict();prof=profiles[profiles.candidate_name.eq(name)].iloc[0];h=history[history.candidate_key.eq(r['candidate_key'])].iloc[0]
 if name=='林于凱':assert '2026' in prof.role_2026 and all(w in norm(prof.education_raw) for w in ['中山大學','生物科學','機械']) and '第三屆高雄市議員' in prof.experience_raw
 else:assert '復興航空' in prof.experience_raw and all(w in prof.education_raw for w in ['七賢','高雄師範','臺東大學'])
 mask=council.candidate_key.eq(r['candidate_key']);status='2022_bulletin_cells_profile_biography_and_current_election_context_supported'
 council.loc[mask,['birth_date_supported_history','birth_year_supported_history','sex_supported_history','birth_sex_evidence_status','history_source_file','history_source_row','history_identity_review','education_2022_declared','first_time_this_office_supported','first_time_any_office_supported','bulletin_identity_status']]=[r['birth_date_declared'],r['birth_year_declared'],r['sex_raw'].strip(),status,h.history_file,h.history_row,status,h.history_education,'0','0',status]
 sources=r['source_url']+' | '+prof.profile_url+((' | '+media['url']) if name=='黃敬雅' else '')
 for field,value in [('birth_date',r['birth_date_declared']),('sex',r['sex_raw'].strip()),('first_time_this_office','0'),('first_time_any_office','0')]:update(r['candidate_key'],field,value,sources,'官方歷史公報及人物履歷／當屆情境支持','來源層級與年代保留；不是戶籍或學位認證；2022已參選，0=非首次。')
 cells.loc[cells.candidate_key.eq(r['candidate_key']),'identity_status']=status
 for item in evidence:
  if item['candidate_key']==r['candidate_key']:item['identity_status']=status
 ev(r['candidate_key'],name,'identity_context',prof.role_2026+' | '+prof.area_2026+' | '+prof.experience_raw,prof.profile_url,prof.source_file,prof.source_sha256,'candidate-specific role/area/qualifications; biography concordance',prof.source_year,status)
 if name=='黃敬雅':
  start=text.index('復興航空');snippet=text[start:start+650].replace('\n','\\n');ev(r['candidate_key'],name,'identity_context',snippet,media['url'],media['path'],media['sha256'],'2026-03-05 interview paragraph: airline career, prior 2022 race and 2026 primary','2026',status)
# Add Meta accounts only from candidate-specific profile blocks accepted above, preserving old evidence.
def canonical(url):
 u=urlparse(url);host=u.netloc.lower().removeprefix('www.');host={'fb.com':'facebook.com','threads.net':'threads.com'}.get(host,host)
 query=urlencode({'id':parse_qs(u.query)['id'][0]}) if host=='facebook.com' and u.path=='/profile.php' and 'id' in parse_qs(u.query) else ''
 return urlunparse(('https','www.'+host,u.path.rstrip('/'),'',query,''))
platforms={'facebook.com':'Facebook','instagram.com':'Instagram','threads.net':'Threads','threads.com':'Threads'};new_social=[]
for r in social_candidates:
 host=urlparse(r['url']).netloc.lower().removeprefix('www.');platform=platforms.get(host,'')
 if not platform:continue
 assert not re.search(r'/(share|sharer|groups|posts|post|reel|reels)(/|$|\?)',urlparse(r['url']).path)
 new_social.append(dict(candidate_key=r['candidate_key'],candidate_name=r['candidate_name'],county_city=r['county_city'],office_type=r['office_type'],platform=platform,account_url=canonical(r['url']),source_url=r['source_url'],source_file=r['source_file'],source_locator='ul.candidate-social-links',evidence_status='candidate_specific_profile_self_link',target_content_status='not_verified_not_posts_captured',checked_date='2026-10-19',source_year=r['source_year'],source_identity_status=r['source_identity_status']))
accounts=pd.concat([accounts,pd.DataFrame(new_social)],ignore_index=True).fillna('').drop_duplicates(['candidate_key','platform','account_url','source_url']).sort_values(['county_city','candidate_name','platform','account_url'])
work=[]
for p in master.to_dict('records'):
 for platform in ['Facebook','Instagram','Threads']:
  d=accounts[accounts.candidate_key.eq(p['candidate_key'])&accounts.platform.eq(platform)];urls=' | '.join(dict.fromkeys(d.account_url));sources=' | '.join(dict.fromkeys(d.source_url))
  row=oldwork[oldwork.candidate_key.eq(p['candidate_key'])&oldwork.platform.eq(platform)].iloc[0].to_dict();row.update(official_urls=urls,discovered_on=sources,account_status='candidate_specific_source_link_target_content_pending' if urls else 'not_yet_found_not_absent');work.append(row)
  if urls:update(p['candidate_key'],platform,urls,sources,'人物頁／競選頁已列帳號連結；目標內容待核對','新來源年代另存社群證據表；歷史連結不宣稱現今有效，未蒐集貼文。')
work=pd.DataFrame(work);assert len(work)==4749 and not work.duplicated(['candidate_key','platform']).any()
# Unrelated requirements are checked over their full candidate scope.
for field in ['incumbency','posts_Facebook','posts_Instagram','posts_Threads','followers','qualification','administrative_experience','political_family']:
 pd.testing.assert_frame_equal(old_f[old_f.field.eq(field)].reset_index(drop=True),fields[fields.field.eq(field)].reset_index(drop=True))
assert len(council)==1502 and council.candidate_key.is_unique and len(master)==1583 and len(fields)==31660
assert council.birth_date_supported_history.ne('').sum()==692 and council.sex_supported_history.ne('').sum()==692
assert len(new_profiles)==7
assert not accounts.account_url.str.contains('/share/|/groups/|/posts/|/post/|/reel/').any()
assert work[['window_count','followers_start','followers_end']].eq('').all().all()
assert profiles[profiles.candidate_name.eq('陳冠宇')].iloc[0].accepted_for_source_inventory=='False'
assert profiles[profiles.candidate_name.eq('李文')].iloc[0].identity_status=='excluded_placeholder_content'
pending=history[~history.candidate_key.isin(cells[cells.candidate_name.isin(['林于凱','黃敬雅'])].candidate_key)].copy();assert len(pending)==162
outputs={'data/processed/snapshots/pre_identity_round7_20261019/council_history_pending_identity_current.csv':pending,'data/processed/council_bulletin_cells_round5_20261019.csv':cells,'data/processed/pridewatch_profile_review_round5_20261019.csv':profiles,'data/processed/candidate_background_evidence_round5_20261019.csv':pd.DataFrame(evidence),'data/processed/snapshots/pre_career_round6_20261019/candidate_master_current.csv':master,'data/processed/snapshots/pre_career_round6_20261019/candidate_fields_current.csv':fields,'data/processed/snapshots/pre_career_round6_20261019/council_background_20261019.csv':council,'data/processed/snapshots/pre_career_round6_20261019/candidate_social_account_evidence_current.csv':accounts,'data/processed/snapshots/pre_career_round6_20261019/meta_candidate_account_worklist_current.csv':work}
verification={}
for path,frame in outputs.items():
 target=(ROOT/path).resolve();assert target.is_relative_to((ROOT/'data/processed').resolve()) and path not in PINS
 frame.to_csv(target,index=False,encoding='utf-8-sig');saved=read(path);assert len(saved)==len(frame) and list(saved.columns)==list(frame.columns)
 assert len(target.read_text(encoding='utf-8-sig').splitlines())==len(saved)+1,path
 verification[path]={'rows':len(saved),'sha256':sha(target)}
summary={'bulletin_candidate_cells_read':13,'new_profile_people':len(new_profiles),'remaining_profile_gaps':int((council.moi_profile_url.eq('')&council.campaign_profile_url.eq('')&council.additional_profile_url.eq('')).sum()),'council_supported_history_birth_dates':692,'council_supported_history_sex':692,'social_people':int(accounts.candidate_key.nunique()),'social_candidate_platform_pairs':len(accounts[['candidate_key','platform']].drop_duplicates()),'social_by_platform':accounts[['candidate_key','platform']].drop_duplicates().platform.value_counts().to_dict(),'new_profile_names':council[council.candidate_key.isin(new_profiles)].candidate_name.tolist(),'outputs':verification}
(ROOT/'data/code/council_bulletin_round5_20261019_verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2));print(json.dumps(summary,ensure_ascii=False,indent=2))

# Regenerate the user's browseable current link list from the exact saved account evidence.
def links(d):return '、'.join(f'[{i+1}]({url})' for i,url in enumerate(dict.fromkeys(d.account_url))) or '待補查'
lines=['# 候選人社群入口（2026-10-19）','','本清單為來源頁列出的可點選帳號入口；尚未全面核對目標帳號內容，未蒐集完整貼文。未找到不代表沒有帳號。舊人物頁連結保留來源年代，不宣稱現今有效。', '', f"目前涵蓋 **{summary['social_people']} 位候選人、{summary['social_candidate_platform_pairs']} 組候選人×平台**。Facebook {summary['social_by_platform']['Facebook']}位、Instagram {summary['social_by_platform']['Instagram']}位、Threads {summary['social_by_platform']['Threads']}位。來源範圍不均，不能用已找到名單比較黨派使用率。",'', '## 蔣萬安、沈伯洋與薛兆基','']
for name in ['蔣萬安','沈伯洋','薛兆基']:
 p=master[master.candidate_name.eq(name)].iloc[0];d=accounts[accounts.candidate_key.eq(p.candidate_key)]
 lines += [f'### {name}（{p.county_city}／{p.office_type}）','','| Facebook | Instagram | Threads |','| --- | --- | --- |','| '+' | '.join(links(d[d.platform.eq(platform)]) for platform in ['Facebook','Instagram','Threads'])+' |','','來源：'+'、'.join(f'[{i+1}]({url})' for i,url in enumerate(dict.fromkeys(d.source_url))),'']
lines+=['## 其他已找到入口的候選人','','| 候選人 | 縣市／公職 | Facebook | Instagram | Threads | 來源 |','| --- | --- | --- | --- | --- | --- |']
for key,d in accounts.groupby('candidate_key',sort=False):
 p=master[master.candidate_key.eq(key)].iloc[0]
 if p.candidate_name in ['蔣萬安','沈伯洋','薛兆基']:continue
 sources='、'.join(f'[{i+1}]({url})' for i,url in enumerate(dict.fromkeys(d.source_url)))
 lines.append('| '+' | '.join([p.candidate_name,p.county_city+'／'+p.office_type]+[links(d[d.platform.eq(platform)]) for platform in ['Facebook','Instagram','Threads']]+[sources])+' |')
lines+=['','## 核對界線','','- 新增PrideWatch連結只取候選人專屬社群區塊，不取組織頁尾。2026選區欄與舊人物頁年代分開記錄。','- 薛兆基仍只有Facebook來源入口，Instagram與Threads待補。','- 分享、貼文、Reels、群組不當作個人帳號；舊有未解析表另存。','- 竹北陳冠宇人物頁未套入高雄同名者；李文占位內容未接受。','- Meta申請、貼文窗口及貼文蒐集依使用者指示暫緩。','- [目前社群工作單](../data/processed/snapshots/pre_career_round6_20261019/meta_candidate_account_worklist_current.csv)、[逐來源帳號證據](../data/processed/snapshots/pre_career_round6_20261019/candidate_social_account_evidence_current.csv)。','- [本輪公報與人物頁核對](council_bulletin_round5_20261019.md)。']
lines[2:2] = ['本輪新增入口見[補查清單：15人、28組社群入口](candidate_social_links_followup_20261019.md)（尚未併入下列 current CSV 統計）。', '']
(ROOT/'report/candidate_social_links_pre_career_round6_20261019.md').write_text('\n'.join(lines)+'\n')


{
  "bulletin_candidate_cells_read": 13,
  "new_profile_people": 7,
  "remaining_profile_gaps": 749,
  "council_supported_history_birth_dates": 692,
  "council_supported_history_sex": 692,
  "social_people": 71,
  "social_candidate_platform_pairs": 172,
  "social_by_platform": {
    "Facebook": 67,
    "Instagram": 62,
    "Threads": 43
  },
  "new_profile_names": [
    "林柏勛",
    "黃朝淵",
    "林于凱",
    "黃敬雅",
    "林志豪",
    "陳珍裕",
    "陳鶴文"
  ],
  "outputs": {
    "data/processed/snapshots/pre_identity_round7_20261019/council_history_pending_identity_current.csv": {
      "rows": 162,
      "sha256": "ca2335dd884a62a8d9b6689cb25c7ca1cea32707911e43ce5ef7deffe2cbbe00"
    },
    "data/processed/council_bulletin_cells_round5_20261019.csv": {
      "rows": 13,
      "sha256": "c62a180244de8991791688276dd40552fb054708bb36a838fd5df6fe9315b35c"
    },
    "data/processed/pridewatch_profile_review_round5_20261019.csv": {
      "rows": 31,
      "sha256": "788f47f9dd2f2ae416d78932c317a1cc935995

14973

## Saved output review

In [2]:
display(read('data/processed/council_bulletin_cells_round5_20261019.csv')[['candidate_name','birth_date_declared','sex_raw','education_raw','identity_status']]);display(read('data/processed/pridewatch_profile_review_round5_20261019.csv')[['candidate_name','identity_status','education_raw']])

,candidate_name,birth_date_declared,sex_raw,education_raw,identity_status
0,林慧欣,1985-06-11,女,國立高雄師範大學客家文化\n研究所,2022_bulletin_candidate_cells_read_2026_identi...
1,李柏融,1949-11-02,男,臺南市六信高級中學,2022_bulletin_candidate_cells_read_2026_identi...
2,鄭東元,1959-04-20,男,國立高雄大學經濟管理碩士\n正修科技大學專科部土木工\n程畢業,2022_bulletin_candidate_cells_read_2026_identi...
3,柳淑芳,1960-08-30,女,左營國小、左營國中、岡山\n農工化工科,2022_bulletin_candidate_cells_read_2026_identi...
4,陳若翠,1969-04-29,女,復興國小、獅甲國中、台南\n家專（現改制台南應用科技\n大學）、高雄師範大學教育\n學院碩士。,2022_bulletin_candidate_cells_read_2026_identi...
5,陳惠君,1968-09-04,女,1.五權國小(高雄市)畢\n2.高雄市立五福國中畢\n3.高雄市立高雄女中畢\n4.國立臺灣...,2022_bulletin_candidate_cells_read_2026_identi...
6,童燕珍,1953-11-02,女,一國立高雄師範大學碩士\n二美國舊金山NHU大學\n三台灣省立師範學院\n(二年制)\n四台...,2022_bulletin_candidate_cells_read_2026_identi...
7,林于凱,1981-04-12,男,國立中山大學生物科學系碩\n士\n國立中山大學機械與機電工\n程學系\n新竹科學園區實驗高中...,2022_bulletin_cells_profile_biography_and_curr...
8,蘇致榮,1977-10-29,男,臺北市立大學運動教育碩士\n、輔仁大學體育學系、高英\n工商、鳳山國中、大東國小,2022_bulletin_candidate_cells_read_2026_identi...
9,徐尚賢,1990-09-30,男,義守大學管理學碩士,2022_bulletin_candidate_cells_read_2026_identi...


,candidate_name,identity_status,education_raw
0,李文,excluded_placeholder_content,啦啦啦
1,林柏勛,historical_profile_county_or_distinctive_biogr...,台北縣菁桐國小 | 台北市民權國小 | 台北市介壽國中 | 延平中學 | 東山高中 | 東吳...
2,顏若芳,2026_profile_name_county_district_match_not_de...,臺北市私立靜心小學 | 臺北市立建成國民中學 | 臺北市立百齡高級中學 | 中國文化大學政治...
3,吳怡萱,historical_profile_current_county_context_unco...,世新大學新聞系 | 台大國家發展研究所（在學中）
4,陳聖文,no_background_fields_present,
5,山田摩衣,2026_profile_name_county_district_match_not_de...,政大國際事務學院碩士
6,卓冠廷,historical_profile_current_county_context_unco...,美國柏克萊大學公共事務碩士 | 國立政治大學公共行政學系 | 板橋高中 | 辭修國中 | 鶯歌國小
7,林秉宥,historical_profile_current_county_context_unco...,東吳大學政治系 | 淡江大學法學碩士
8,陳宜,2026_profile_name_county_district_match_not_de...,崇右技術學院應用外文系畢業 | 台北市立大學競技運動訓練研究所碩士畢業
9,林佩螢,historical_profile_current_county_context_unco...,蘭陽技術學院 建築科 副學士 | 蘭陽技術學院 建築系 學士 | 臺灣科技大學 建築所 碩士
